# ABLA — Controlled Ablation of Input Baseline Centering

This notebook tests one question: **How does sensor scaling affect multi-temperature LOMO generalization and input-to-reservoir dynamics?**

Three otherwise identical pipelines are compared:

- **Not baseline-centered:** the smoothed sensor levels enter the ESN directly.
- **Baseline-centered:** each smoothed sensor channel is replaced by $u(t)-u(0)$.
- **Per-trial normalized:** each sensor is independently mapped so its maximum is 0 and minimum is −1.

All cases use the same 0–5 s window, summarized ESN trajectories, LOMO folds, reservoir realizations, and fixed ESN–XGBoost hyperparameters. The XGBoost regressor receives **summarized ESN features only**—no manually computed thermal features and no temperature predictor. No Bayesian or Optuna search occurs. The purpose is controlled comparison, not maximum achievable performance.

# 1. Imports and fixed reference configuration

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import linalg
from scipy.signal import savgol_filter
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


In [ ]:
TEMPERATURE_FOLDERS = ("30C", "40C", "50C", "60C")
TARGET = "eff"
ANALYSIS_WINDOW = (0.0, 5.0)
# Predeclared semi-universal reference settings; these are not optimized here.
ESN_PARAMS = {
    "res_size": 20,
    "leak_rate": 0.30,
    "input_magnitude": 1.00,
    "spectral_radius": 0.90,
    "washout": 0,
}
XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "min_child_weight": 5,
    "reg_alpha": 0.10,
    "reg_lambda": 10.0,
}
RESERVOIR_SEEDS = (42, 43, 44)
RANDOM_STATE = 42
CLIP_PREDICTIONS_TO_TRAIN_RANGE = True

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, cwd.parent) if (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")
DATA_ROOT = PROJECT_ROOT / "data" / "02_preprocessed"
RESULTS_DIR = PROJECT_ROOT / "results" / "ablation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULT_STEM = "ABLA_three_input_scalings_fixed_HP_0to5_summarized_ESN_only_LOMO"

print("Fixed ESN HPs:", ESN_PARAMS)
print("Fixed XGBoost HPs:", XGB_PARAMS)
print("Reservoir seeds:", RESERVOIR_SEEDS)


## Why these HPs are fixed

- **20 units:** enough nonlinear response modes while limiting dimensionality.
- **Leak rate 0.30:** moderate updating that balances responsiveness and memory.
- **Input magnitude 1.00:** neutral reference strength for standardized inputs.
- **Spectral radius 0.90:** recurrent memory near but below the conventional stability boundary.
- **Washout 0:** contact is aligned to $t=0$ and the initial transient must be retained.
- **Three seeds:** reduces dependence on one random reservoir realization.
- **Shallow regularized XGBoost:** permits nonlinear relationships while discouraging excessive tree complexity.

# 2. Load trials and apply standard material properties

In [ ]:
STANDARD_PROPERTIES = {
    "ps_foam":  {"k": 0.034, "rho": 25.0,   "cp": 1400.0},
    "pu_foam":  {"k": 0.043, "rho": 30.0,   "cp": 1400.0},
    "cork":     {"k": 0.043, "rho": 240.0,  "cp": 1800.0},
    "wood":     {"k": 0.150, "rho": 700.0,  "cp": 1700.0},
    "pdms":     {"k": 0.150, "rho": 970.0,  "cp": 1460.0},
    "gypsum":   {"k": 0.170, "rho": 800.0,  "cp": 1090.0},
    "cement":   {"k": 0.290, "rho": 1440.0, "cp": 750.0},
    "graphite": {"k": 100.0, "rho": 1820.0, "cp": 710.0},
    "bismuth":  {"k": 8.1,   "rho": 9780.0, "cp": 130.0},
    "titanium": {"k": 21.9,  "rho": 4506.0, "cp": 523.0},
    "nickel":   {"k": 90.9,  "rho": 8908.0, "cp": 461.0},
    "iron":     {"k": 80.4,  "rho": 7874.0, "cp": 449.0},
    "aluminum": {"k": 237.0, "rho": 2700.0, "cp": 897.0},
    "copper":   {"k": 401.0, "rho": 8960.0, "cp": 385.0},
}
SAMPLE_ALIASES = {
    "ps": "ps_foam", "ps foam": "ps_foam", "ps_foam": "ps_foam",
    "pu": "pu_foam", "pu foam": "pu_foam", "pu_foam": "pu_foam",
    "cork": "cork", "cork fine": "cork", "cork_fine": "cork",
    "wood": "wood", "pdms": "pdms", "gypsum": "gypsum",
    "cement": "cement", "graphite": "graphite", "carbon": "graphite",
    "bi": "bismuth", "bismuth": "bismuth",
    "ti": "titanium", "titanium": "titanium",
    "ni": "nickel", "nickel": "nickel",
    "fe": "iron", "iron": "iron",
    "al": "aluminum", "aluminum": "aluminum",
    "cu": "copper", "copper": "copper",
}

required = {"Sample", "Trial", "Time", "Primary", "Secondary"}
frames = []
for temperature in TEMPERATURE_FOLDERS:
    for path in sorted((DATA_ROOT / temperature).glob("*.csv")):
        frame = pd.read_csv(path)
        if required - set(frame.columns):
            warnings.warn(f"Skipping {path.name}: missing required columns")
            continue
        frame["Temperature"] = temperature
        frames.append(frame)
if not frames:
    raise ValueError("No valid preprocessed trial files were found.")

DATA = pd.concat(frames, ignore_index=True).replace([np.inf, -np.inf], np.nan)
for column in ("Trial", "Time", "Primary", "Secondary"):
    DATA[column] = pd.to_numeric(DATA[column], errors="coerce")
DATA = DATA.dropna(subset=list(required) + ["Temperature"]).copy()
DATA["Trial"] = DATA["Trial"].astype(int)
normalized = DATA["Sample"].astype(str).str.strip().str.lower().str.replace("_", " ")
DATA["Sample"] = normalized.map(SAMPLE_ALIASES)
if DATA["Sample"].isna().any():
    raise KeyError(f"Unknown materials: {sorted(normalized[DATA['Sample'].isna()].unique())}")
DATA["Temperature_C"] = pd.to_numeric(DATA["Temperature"].str.extract(r"(\d+)", expand=False))
for name in ("k", "rho", "cp"):
    DATA[name] = DATA["Sample"].map(lambda material: STANDARD_PROPERTIES[material][name])
DATA["eff"] = np.sqrt(DATA["k"] * DATA["rho"] * DATA["cp"])
DATA["trial_id"] = (DATA["Temperature"] + "__" + DATA["Sample"] + "_trial_" + DATA["Trial"].astype(str))
DATA = DATA.sort_values(["trial_id", "Time"]).reset_index(drop=True)
print(f"Loaded {DATA['trial_id'].nunique()} temperature-specific trials.")


# 3. Detect contact and retain the common 0–5 s response

In [ ]:
def find_contact_time(trial, smooth_window=15, polyorder=2, threshold_frac=0.30, skip_samples=5):
    clean = (trial[["Time", "Primary"]].dropna().sort_values("Time")
             .drop_duplicates("Time").reset_index(drop=True))
    time = clean["Time"].to_numpy(float)
    signal = clean["Primary"].to_numpy(float)
    if len(signal) < skip_samples + 7 or np.any(np.diff(time) <= 0):
        raise ValueError("Insufficient or invalid samples.")
    work_time, work_signal = time[skip_samples:], signal[skip_samples:]
    window = min(smooth_window, len(work_signal))
    if window % 2 == 0: window -= 1
    smooth = savgol_filter(work_signal, window, polyorder, mode="interp")
    derivative = np.gradient(smooth, work_time)
    strongest = int(np.argmin(derivative))
    active = derivative < threshold_frac * derivative[strongest]
    elbow = 0
    for position in range(strongest, -1, -1):
        if not active[position]:
            elbow = position + 1
            break
    return float(work_time[elbow])

aligned_trials = {}
metadata_rows = []
for trial_id, trial in DATA.groupby("trial_id", sort=False):
    trial = trial.sort_values("Time").drop_duplicates("Time").copy()
    try:
        contact = find_contact_time(trial)
    except ValueError as exc:
        warnings.warn(f"Skipping {trial_id}: {exc}")
        continue
    trial["time_from_contact"] = trial["Time"] - contact
    trial = trial[trial["time_from_contact"].between(*ANALYSIS_WINDOW)].copy()
    if len(trial) < 10:
        warnings.warn(f"Skipping {trial_id}: fewer than 10 retained samples")
        continue
    aligned_trials[trial_id] = trial.reset_index(drop=True)
    metadata_rows.append({
        "trial_id": trial_id, "Temperature": trial["Temperature"].iloc[0],
        "Temperature_C": float(trial["Temperature_C"].iloc[0]),
        "Sample": trial["Sample"].iloc[0], "Trial": int(trial["Trial"].iloc[0]),
        "eff": float(trial["eff"].iloc[0]),
    })
METADATA = pd.DataFrame(metadata_rows).sort_values("trial_id").reset_index(drop=True)
print(f"Aligned trials retained: {len(METADATA)}")
display(METADATA.groupby("Temperature").agg(trials=("trial_id", "size"), materials=("Sample", "nunique")))


# 4. Three condition-specific ESN input representations

In [ ]:
def smooth_signal(values, window=11, polyorder=2):
    values = np.asarray(values, float)
    selected = min(window, len(values))
    if selected % 2 == 0: selected -= 1
    return savgol_filter(values, selected, polyorder, mode="interp") if selected >= 5 else values.copy()

def normalize_max_zero_min_minus_one(values):
    """Map one complete trial channel to [-1, 0] without mixing sensors."""
    values = np.asarray(values, float)
    minimum, maximum = float(np.min(values)), float(np.max(values))
    span = maximum - minimum
    if not np.isfinite(span) or np.isclose(span, 0):
        raise ValueError("Cannot min–max normalize a constant or invalid sensor channel.")
    return (values - maximum) / span

CONDITIONS = {
    "A_not_baseline_centered": "raw",
    "B_baseline_centered": "baseline_centered",
    "C_per_trial_minmax_-1_to_0": "minmax_-1_to_0",
}

def esn_input(trial, input_mode):
    time = trial["time_from_contact"].to_numpy(float)
    primary = smooth_signal(trial["Primary"])
    secondary = smooth_signal(trial["Secondary"])
    if input_mode == "baseline_centered":
        primary = primary - primary[0]
        secondary = secondary - secondary[0]
    elif input_mode == "minmax_-1_to_0":
        primary = normalize_max_zero_min_minus_one(primary)
        secondary = normalize_max_zero_min_minus_one(secondary)
    elif input_mode != "raw":
        raise ValueError(f"Unknown input mode: {input_mode}")
    return np.column_stack([
        primary, secondary, primary-secondary,
        np.gradient(primary, time), np.gradient(secondary, time),
    ])


# 5. Fixed reservoir and nonredundant trajectory summaries

In [ ]:
class ManualReservoir:
    def __init__(self, random_state):
        self.res_size = ESN_PARAMS["res_size"]
        self.leak_rate = ESN_PARAMS["leak_rate"]
        self.washout = ESN_PARAMS["washout"]
        rng = np.random.default_rng(random_state)
        self.Win = (rng.random((self.res_size, 6)) - 0.5) * ESN_PARAMS["input_magnitude"]
        W = rng.random((self.res_size, self.res_size)) - 0.5
        radius = np.max(np.abs(linalg.eigvals(W)))
        self.W = (W / radius.real) * ESN_PARAMS["spectral_radius"]

    def run(self, sequence):
        x = np.zeros((self.res_size, 1))
        states = []
        for row in np.asarray(sequence, float):
            proposed = np.tanh(self.Win @ np.r_[1.0, row].reshape(-1, 1) + self.W @ x)
            x = (1-self.leak_rate)*x + self.leak_rate*proposed
            states.append(x[:, 0].copy())
        return np.asarray(states)[self.washout:]

def summarize_states(time, states):
    time = np.asarray(time, float)
    states = np.asarray(states, float)
    output = {}
    for unit in range(states.shape[1]):
        x = states[:, unit]
        prefix = f"esn_u{unit:03d}"
        output[f"{prefix}_mean"] = float(np.mean(x))
        output[f"{prefix}_std"] = float(np.std(x))
        output[f"{prefix}_min"] = float(np.min(x))
        output[f"{prefix}_max"] = float(np.max(x))
        output[f"{prefix}_initial"] = float(x[0])
        output[f"{prefix}_net_change"] = float(x[-1] - x[0])
        output[f"{prefix}_slope"] = float(np.polyfit(time, x, 1)[0])
        output[f"{prefix}_mean_abs"] = float(np.mean(np.abs(x)))
        output[f"{prefix}_abs_auc"] = float(np.trapezoid(np.abs(x), time))
    return output

print(f"ESN summary predictors per seed: {ESN_PARAMS['res_size'] * 9}")


# 6. Leakage-safe paired LOMO evaluation

Within each material fold, a separate five-channel scaler is fitted for each input-scaling condition using only the 312 training trials. All three conditions use exactly the same held-out material and the same reservoir matrices for each seed.

In [ ]:
def make_regressor(random_state):
    xgb = XGBRegressor(
        objective="reg:squarederror", random_state=int(random_state),
        n_jobs=-1, tree_method="hist", verbosity=0, **XGB_PARAMS,
    )
    base = Pipeline([("imputer", SimpleImputer(strategy="median")), ("xgb", xgb)])
    return TransformedTargetRegressor(
        regressor=base, func=np.log1p, inverse_func=np.expm1, check_inverse=False
    )

def pooled_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    target_range = float(np.ptp(y_true))
    return {
        "n": len(y_true), "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse, "nrmse_range": rmse/target_range,
        "r2": float(r2_score(y_true, y_pred)),
        "median_ape_pct": float(np.median(np.abs((y_true-y_pred)/y_true))*100),
    }

def feature_table(train_ids, all_ids, input_mode, seed):
    scaler = StandardScaler().fit(np.vstack([
        esn_input(aligned_trials[trial_id], input_mode) for trial_id in train_ids
    ]))
    reservoir = ManualReservoir(seed)
    rows = []
    for trial_id in all_ids:
        trial = aligned_trials[trial_id]
        sequence = scaler.transform(esn_input(trial, input_mode))
        states = reservoir.run(sequence)
        time = trial["time_from_contact"].to_numpy(float)[ESN_PARAMS["washout"]:]
        row = summarize_states(time, states)
        row["trial_id"] = trial_id
        rows.append(row)
    return pd.DataFrame(rows).set_index("trial_id")

logo = LeaveOneGroupOut()
prediction_rows, fold_rows = [], []
for fold, (train_idx, test_idx) in enumerate(logo.split(METADATA, groups=METADATA["Sample"]), start=1):
    train_meta, test_meta = METADATA.iloc[train_idx], METADATA.iloc[test_idx]
    train_ids, test_ids = train_meta["trial_id"].tolist(), test_meta["trial_id"].tolist()
    y_train, y_test = train_meta[TARGET].to_numpy(float), test_meta[TARGET].to_numpy(float)
    material = test_meta["Sample"].iloc[0]
    print(f"Fold {fold:02d}/14: held out {material}")

    for condition, input_mode in CONDITIONS.items():
        seed_predictions = []
        for seed in RESERVOIR_SEEDS:
            features = feature_table(train_ids, train_ids + test_ids, input_mode, seed)
            X_train = features.loc[train_ids].reset_index(drop=True)
            X_test = features.loc[test_ids].reset_index(drop=True)
            model = make_regressor(seed)
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            if CLIP_PREDICTIONS_TO_TRAIN_RANGE:
                pred = np.clip(pred, y_train.min(), y_train.max())
            seed_predictions.append(pred)
        prediction = np.mean(seed_predictions, axis=0)
        residual = prediction - y_test
        rmse = float(np.sqrt(np.mean(residual**2)))
        fold_rows.append({
            "condition": condition, "fold": fold, "held_out_material": material,
            "n": len(y_test), "mae": float(np.mean(np.abs(residual))), "rmse": rmse,
            "nrmse_training_range": rmse/float(np.ptp(y_train)),
            "mean_error_bias": float(np.mean(residual)),
            "median_ape_pct": float(np.median(np.abs(residual/y_test))*100),
        })
        for row_index, (_, meta_row) in enumerate(test_meta.iterrows()):
            prediction_rows.append({
                "condition": condition, "fold": fold, "trial_id": meta_row["trial_id"],
                "Temperature": meta_row["Temperature"], "Sample": meta_row["Sample"],
                "Trial": meta_row["Trial"], "y_true": y_test[row_index],
                "y_pred": prediction[row_index],
            })

OOF = pd.DataFrame(prediction_rows)
FOLD_METRICS = pd.DataFrame(fold_rows)
OVERALL = pd.DataFrame([
    {"condition": condition, **pooled_metrics(part["y_true"], part["y_pred"])}
    for condition, part in OOF.groupby("condition", sort=False)
])
display(OVERALL)


# 7. Paired performance comparison

In [ ]:
paired_folds = FOLD_METRICS.pivot(index="held_out_material", columns="condition", values=["rmse", "nrmse_training_range", "median_ape_pct"])
paired_folds[("delta", "B_centered_minus_A_raw")] = (
    paired_folds[("nrmse_training_range", "B_baseline_centered")]
    - paired_folds[("nrmse_training_range", "A_not_baseline_centered")]
)
paired_folds[("delta", "C_minmax_minus_A_raw")] = (
    paired_folds[("nrmse_training_range", "C_per_trial_minmax_-1_to_0")]
    - paired_folds[("nrmse_training_range", "A_not_baseline_centered")]
)
display(paired_folds.sort_values(("delta", "B_centered_minus_A_raw")))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
summary_plot = OVERALL.set_index("condition")
summary_plot[["nrmse_range"]].plot(kind="bar", legend=False, ax=axes[0], color="#377eb8")
axes[0].set(title="Pooled LOMO NRMSE", ylabel="NRMSE (lower is better)", xlabel="")
summary_plot[["r2"]].plot(kind="bar", legend=False, ax=axes[1], color="#4daf4a")
axes[1].set(title="Pooled LOMO R²", ylabel="R² (higher is better)", xlabel="")
plt.tight_layout(); plt.show()

deltas = paired_folds["delta"].sort_index()
ax = deltas.plot(kind="barh", figsize=(10, 7), color=["#188F7A", "#9467bd"])
ax.axvline(0, color="black", linewidth=1)
ax.set(title="Paired material effects relative to Condition A", xlabel="Δ training-range NRMSE relative to uncentered input")
plt.tight_layout(); plt.show()
print("Negative Δ means that preprocessing condition improved the held-out material relative to Condition A.")


# 8. Input-to-dynamics comparison for a customizable held-out material

In [ ]:
DIAGNOSTIC_MATERIAL = "aluminum"
DIAGNOSTIC_TEMPERATURE = "40C"
DIAGNOSTIC_REPETITION = 1
DIAGNOSTIC_SEED = 42
REPRESENTATIVE_UNITS = (0, 3, 7, 11, 15, 19)

selected = METADATA[(METADATA["Sample"] == DIAGNOSTIC_MATERIAL) & (METADATA["Temperature"] == DIAGNOSTIC_TEMPERATURE) & (METADATA["Trial"] == DIAGNOSTIC_REPETITION)]
if selected.empty:
    raise KeyError("The requested diagnostic trial was not found.")
trial_id = selected["trial_id"].iloc[0]
training_ids = METADATA.loc[METADATA["Sample"] != DIAGNOSTIC_MATERIAL, "trial_id"].tolist()
trial = aligned_trials[trial_id]
time = trial["time_from_contact"].to_numpy(float)

diagnostics = {}
for condition, input_mode in CONDITIONS.items():
    scaler = StandardScaler().fit(np.vstack([esn_input(aligned_trials[x], input_mode) for x in training_ids]))
    scaled = scaler.transform(esn_input(trial, input_mode))
    states = ManualReservoir(DIAGNOSTIC_SEED).run(scaled)
    diagnostics[condition] = {"scaled_inputs": scaled, "states": states}

fig, axes = plt.subplots(2, 3, figsize=(20, 9), sharex=True)
channel_names = ["Primary", "Secondary", "Primary−Secondary", "dPrimary/dt", "dSecondary/dt"]
for column, (condition, values) in enumerate(diagnostics.items()):
    for channel, name in enumerate(channel_names):
        axes[0, column].plot(time, values["scaled_inputs"][:, channel], label=name, linewidth=1.2)
    axes[0, column].set(title=condition.replace("_", " "), ylabel="Training-fold z-score")
    axes[0, column].legend(fontsize=8, ncol=2)
    for unit in REPRESENTATIVE_UNITS:
        axes[1, column].plot(time, values["states"][:, unit], linewidth=1.2, label=f"Unit {unit}")
    axes[1, column].set(xlabel="Time from contact (s)", ylabel="Activation")
    axes[1, column].legend(fontsize=8, ncol=3)
fig.suptitle(f"Input → ESN dynamics: {trial_id}", fontsize=14)
plt.tight_layout(); plt.show()

activity_rows = []
for condition, values in diagnostics.items():
    states = values["states"]
    activity_rows.append({
        "condition": condition,
        "mean_rms_activation": float(np.mean(np.sqrt(np.mean(states**2, axis=1)))),
        "mean_absolute_activation": float(np.mean(np.abs(states))),
        "fraction_near_saturation_abs_gt_0.95": float(np.mean(np.abs(states) > 0.95)),
    })
display(pd.DataFrame(activity_rows))


# 9. Actual-versus-predicted and material-mean behavior

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True, sharey=True)
for ax, (condition, part) in zip(axes, OOF.groupby("condition", sort=False)):
    for material, material_part in part.groupby("Sample"):
        ax.scatter(material_part["y_true"], material_part["y_pred"], alpha=0.65, label=material)
    low = min(part["y_true"].min(), part["y_pred"].min())
    high = max(part["y_true"].max(), part["y_pred"].max())
    ax.plot([low, high], [low, high], "k:")
    ax.set(xscale="log", yscale="log", title=condition.replace("_", " "), xlabel="Actual effusivity", ylabel="OOF prediction")
axes[1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

MATERIAL_MEANS = OOF.groupby(["condition", "Sample"], as_index=False).agg(y_true=("y_true", "mean"), y_pred=("y_pred", "mean"), prediction_sd=("y_pred", "std"), n=("trial_id", "size"))
MATERIAL_LEVEL_RESULTS = pd.DataFrame([
    {"condition": condition, **pooled_metrics(part["y_true"], part["y_pred"])}
    for condition, part in MATERIAL_MEANS.groupby("condition", sort=False)
])
print("LOMO performance after averaging 24 OOF trials per material:")
display(MATERIAL_LEVEL_RESULTS)


# 10. Selected-material input → ESN dynamics → LOMO OOF correspondence

This reproduces the requested presentation style for any one preprocessing condition. Each material block shows its actual and OOF-predicted effusivity, condition-specific sensor representation, and representative reservoir trajectories. Lines are repetition means and shading is ±1 repetition SD.

In [ ]:
CASE_STUDY_CONDITION = "B_baseline_centered"  # choose any key in CONDITIONS
CASE_STUDY_TEMPERATURE = "60C"
CASE_STUDY_MATERIALS = ["gypsum", "wood", "pdms", "cement"]
CASE_STUDY_SEED = 42
CASE_STUDY_UNITS = (0, 3, 7, 11, 15, 19)
CASE_STUDY_GRID_POINTS = 101

if CASE_STUDY_CONDITION not in CONDITIONS:
    raise KeyError(f"Unknown condition. Choose from {list(CONDITIONS)}")
case_mode = CONDITIONS[CASE_STUDY_CONDITION]
case_grid = np.linspace(*ANALYSIS_WINDOW, CASE_STUDY_GRID_POINTS)
block_length = ANALYSIS_WINDOW[1] - ANALYSIS_WINDOW[0]
gap = 0.35

fig, axes = plt.subplots(3, 1, figsize=(18, 13), sharex=True, gridspec_kw={"height_ratios": [1.0, 1.2, 1.4]})
material_labels = []
unit_colors = plt.cm.tab10(np.linspace(0, 1, len(CASE_STUDY_UNITS)))
for block, material in enumerate(CASE_STUDY_MATERIALS):
    selected_meta = METADATA[(METADATA["Sample"] == material) & (METADATA["Temperature"] == CASE_STUDY_TEMPERATURE)].sort_values("Trial")
    if selected_meta.empty:
        warnings.warn(f"No trials found for {material} at {CASE_STUDY_TEMPERATURE}")
        continue
    selected_ids = selected_meta["trial_id"].tolist()
    x_offset = block * (block_length + gap)
    x = x_offset + case_grid

    # Trial-level LOMO predictions already produced by the paired evaluation.
    prediction_part = OOF[(OOF["condition"] == CASE_STUDY_CONDITION) & (OOF["Sample"] == material) & (OOF["Temperature"] == CASE_STUDY_TEMPERATURE)]
    actual = float(selected_meta[TARGET].iloc[0])
    predicted_mean = float(prediction_part["y_pred"].mean())
    predicted_sd = float(prediction_part["y_pred"].std(ddof=1))
    axes[0].plot([x[0], x[-1]], [actual, actual], color="black", linewidth=2.0, label="Actual eff" if block == 0 else None)
    axes[0].plot([x[0], x[-1]], [predicted_mean, predicted_mean], color="#1f77b4", linestyle="--", linewidth=2.0, label="Mean OOF prediction" if block == 0 else None)
    axes[0].fill_between(x, max(predicted_mean-predicted_sd, 1e-12), predicted_mean+predicted_sd, color="#1f77b4", alpha=0.16)

    # Recreate this material's leakage-safe LOMO input scaler.
    fold_train_ids = METADATA.loc[METADATA["Sample"] != material, "trial_id"].tolist()
    fold_scaler = StandardScaler().fit(np.vstack([esn_input(aligned_trials[t], case_mode) for t in fold_train_ids]))
    sensor_trials = []
    state_trials = []
    for selected_id in selected_ids:
        selected_trial = aligned_trials[selected_id]
        original_time = selected_trial["time_from_contact"].to_numpy(float)
        represented = esn_input(selected_trial, case_mode)
        sensor_trials.append(np.column_stack([
            np.interp(case_grid, original_time, represented[:, 0]),
            np.interp(case_grid, original_time, represented[:, 1]),
        ]))
        states = ManualReservoir(CASE_STUDY_SEED).run(fold_scaler.transform(represented))
        state_trials.append(np.column_stack([
            np.interp(case_grid, original_time, states[:, unit]) for unit in CASE_STUDY_UNITS
        ]))

    sensor_trials = np.asarray(sensor_trials)
    state_trials = np.asarray(state_trials)
    sensor_mean, sensor_sd = sensor_trials.mean(axis=0), sensor_trials.std(axis=0, ddof=1)
    state_mean, state_sd = state_trials.mean(axis=0), state_trials.std(axis=0, ddof=1)
    for channel, (name, color) in enumerate((("Primary", "#1f77b4"), ("Secondary", "#ff7f0e"))):
        axes[1].plot(x, sensor_mean[:, channel], color=color, linewidth=1.7, label=name if block == 0 else None)
        axes[1].fill_between(x, sensor_mean[:, channel]-sensor_sd[:, channel], sensor_mean[:, channel]+sensor_sd[:, channel], color=color, alpha=0.13)
    for unit_index, unit in enumerate(CASE_STUDY_UNITS):
        color = unit_colors[unit_index]
        axes[2].plot(x, state_mean[:, unit_index], color=color, linewidth=1.4, label=f"Unit {unit}" if block == 0 else None)
        axes[2].fill_between(x, state_mean[:, unit_index]-state_sd[:, unit_index], state_mean[:, unit_index]+state_sd[:, unit_index], color=color, alpha=0.10)

    material_labels.append((x_offset + block_length/2, f"{material}\neff={actual:,.0f}\n(n={len(selected_ids)})"))
    if block < len(CASE_STUDY_MATERIALS)-1:
        boundary = x_offset + block_length + gap/2
        for ax in axes:
            ax.axvline(boundary, color="red", linestyle="--", alpha=0.65)

axes[0].set(yscale="log", ylabel="Effusivity", title=f"Selected-material comparison at {CASE_STUDY_TEMPERATURE}: {CASE_STUDY_CONDITION}")
axes[0].legend(loc="best")
axes[1].set(ylabel="Condition-specific sensor representation", title="Physical inputs: repetition mean ± SD")
axes[1].legend(loc="best")
axes[2].set(ylabel="Activation", title=f"Representative ESN trajectories: repetition mean ± SD, seed={CASE_STUDY_SEED}")
axes[2].legend(ncol=3, fontsize=8)
axes[2].set_xticks([position for position, _ in material_labels], [label for _, label in material_labels])
axes[2].set_xlabel("Independent 0–5 s material trial blocks")
plt.tight_layout(); plt.show()


# 11. Save reproducible ablation outputs and provenance

In [ ]:
OOF.to_csv(RESULTS_DIR / f"{RESULT_STEM}_oof_predictions.csv", index=False)
FOLD_METRICS.to_csv(RESULTS_DIR / f"{RESULT_STEM}_fold_metrics.csv", index=False)
OVERALL.to_csv(RESULTS_DIR / f"{RESULT_STEM}_overall_metrics.csv", index=False)
MATERIAL_MEANS.to_csv(RESULTS_DIR / f"{RESULT_STEM}_material_means.csv", index=False)
MATERIAL_LEVEL_RESULTS.to_csv(RESULTS_DIR / f"{RESULT_STEM}_material_level_metrics.csv", index=False)

provenance = {
    "research_question": "Effect of three sensor scaling representations under fixed model settings",
    "conditions": {
        "A_not_baseline_centered": "u(t)",
        "B_baseline_centered": "u(t)-u(0)",
        "C_per_trial_minmax_-1_to_0": "(u(t)-max(u))/(max(u)-min(u)), separately per sensor and trial",
    },
    "analysis_window": list(ANALYSIS_WINDOW),
    "reservoir_representation": "nine nonredundant statistical summaries per unit",
    "regressor_inputs": ["summarized ESN trajectories only"],
    "esn_parameters": ESN_PARAMS, "xgb_parameters": XGB_PARAMS,
    "reservoir_seeds": list(RESERVOIR_SEEDS),
    "validation": "14-fold LOMO with pooled OOF metrics",
    "clip_predictions_to_training_range": CLIP_PREDICTIONS_TO_TRAIN_RANGE,
}
with (RESULTS_DIR / f"{RESULT_STEM}_provenance.json").open("w") as file:
    json.dump(provenance, file, indent=2)
print("Saved results to:", RESULTS_DIR)


# 12. Interpretation checklist

1. Compare pooled NRMSE and $R^2$; neither metric alone establishes improvement.
2. Inspect paired per-material NRMSE differences. A negative centered-minus-uncentered difference favors centering.
3. Check whether centering or per-trial normalization changes input separation, reservoir activation, or saturation.
4. Check whether any pooled improvement is widespread or driven by only a few high-effusivity materials.
5. Report predictor counts, fixed HPs, folds, and seeds with the results.
6. Conclude only that a preprocessing treatment helped or hurt **under this fixed reference model**; this is not a comparison of independently optimized pipelines.
7. Condition C deliberately removes absolute per-trial amplitude. If it performs worse, amplitude likely carried useful effusivity information; if it performs better, relative trajectory shape may be more useful than absolute magnitude.